# MeatLens Training Notebook
Transfer learning experiments for pork freshness classification using ROI images.

**Backbones**: MobileNetV3Small, EfficientNetB0, ResNet50, MobileNetV2.

**Experiments**: exp1 and exp2 (CSV splits already prepared).
- Train/val/test splits come from CSV files.
- Uses only ROI image files (`roi_file`) as inputs.
- Time is not used as a model input.

In [ ]:
# Core imports
import os
import random
import time
import math
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications import MobileNetV3Small, EfficientNetB0, ResNet50, MobileNetV2
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as preprocess_mobilenetv3
from tensorflow.keras.applications.efficientnet import preprocess_input as preprocess_efficientnet
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_resnet50
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mobilenetv2

# Metrics and reports
try:
    from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score
    HAVE_SKLEARN = True
except Exception:
    HAVE_SKLEARN = False
    print('scikit-learn not available; install if you want classification reports and confusion matrices.')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Paths and settings
DATA_ROOT = Path('.')
SAMPLE_DIRS = {
    '1': 'Pork Shoulder - sample 1',
    '2': 'Pork Shoulder - sample 2',
}

EXP_CSVS = {
    'exp1': {
        'train': 'exp1_train.csv',
        'val': 'exp1_val.csv',
        'test': 'exp1_test.csv',
    },
    'exp2': {
        'train': 'exp2_train.csv',
        'val': 'exp2_val.csv',
        'test': 'exp2_test.csv',
    },
}

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

EPOCHS_FE = 20
EPOCHS_FT = 10
FINE_TUNE_FRACTION = 0.2
LR_FE = 1e-3
LR_FT = 1e-5

RUN_TRAINING = True  # Set False to skip training cells when re-running notebook

In [ ]:
# Data loading helpers
def resolve_image_path(row: pd.Series) -> str:
    roi_file = str(row.get('roi_file', '')).strip()
    if os.path.isabs(roi_file):
        return roi_file
    # If roi_file already includes a relative path, respect it
    if os.sep in roi_file or '/' in roi_file:
        return str(DATA_ROOT / roi_file)
    folder = str(row.get('folder', '')).strip()
    sample = str(row.get('sample_number', '')).strip()
    sample_dir = SAMPLE_DIRS.get(sample, '')
    if sample_dir:
        return str(DATA_ROOT / sample_dir / folder / roi_file)
    if folder:
        return str(DATA_ROOT / folder / roi_file)
    return str(DATA_ROOT / roi_file)


def load_split_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    expected_cols = {'roi_file', 'label', 'sample_number', 'time', 'folder'}
    missing = expected_cols.difference(df.columns)
    if missing:
        raise ValueError(f'Missing columns in {path}: {missing}')
    df = df.copy()
    df['image_path'] = df.apply(resolve_image_path, axis=1)
    return df


def build_label_mapping(dfs: list[pd.DataFrame]):
    labels = sorted(set(pd.concat(dfs)['label'].astype(str)))
    label_to_index = {name: i for i, name in enumerate(labels)}
    return labels, label_to_index


def add_label_index(df: pd.DataFrame, label_to_index: dict) -> pd.DataFrame:
    df = df.copy()
    df['label_idx'] = df['label'].map(label_to_index).astype(int)
    return df


def compute_class_weights(labels: list[int], num_classes: int) -> dict:
    counts = np.bincount(labels, minlength=num_classes)
    total = counts.sum()
    weights = {i: total / (num_classes * c) if c > 0 else 0.0 for i, c in enumerate(counts)}
    return weights, counts


def validate_paths(df: pd.DataFrame, n: int = 5) -> None:
    missing = df[~df['image_path'].apply(os.path.exists)]
    print('Missing files:', len(missing))
    if len(missing) > 0:
        print(missing[['roi_file', 'image_path']].head(n))


# Load all CSVs for label mapping
all_dfs = []
for exp in EXP_CSVS.values():
    for split in exp.values():
        all_dfs.append(load_split_csv(split))

label_names, label_to_index = build_label_mapping(all_dfs)
num_classes = len(label_names)
print('Label mapping:', label_to_index)

In [ ]:
# Image preprocessing and augmentation
def decode_image(path: tf.Tensor) -> tf.Tensor:
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    return img


def random_rotate_180(img: tf.Tensor) -> tf.Tensor:
    do_rotate = tf.random.uniform([]) < 0.5
    return tf.cond(do_rotate, lambda: tf.image.rot90(img, k=2), lambda: img)


def random_sharpness(img: tf.Tensor) -> tf.Tensor:
    factor = tf.random.uniform([], 0.8, 1.2)
    return tf.image.adjust_sharpness(img, factor)


def random_blur(img: tf.Tensor) -> tf.Tensor:
    def _blur():
        kernel = tf.constant([[1, 2, 1], [2, 4, 2], [1, 2, 1]], dtype=tf.float32) / 16.0
        kernel = kernel[:, :, tf.newaxis, tf.newaxis]
        kernel = tf.tile(kernel, [1, 1, 3, 1])
        img4 = tf.expand_dims(img, 0)
        blurred = tf.nn.depthwise_conv2d(img4, kernel, strides=[1, 1, 1, 1], padding='SAME')
        return tf.squeeze(blurred, 0)
    do_blur = tf.random.uniform([]) < 0.3
    return tf.cond(do_blur, _blur, lambda: img)


def add_gaussian_noise(img: tf.Tensor) -> tf.Tensor:
    noise = tf.random.normal(tf.shape(img), mean=0.0, stddev=5.0)
    return img + noise


def augment_image(img: tf.Tensor) -> tf.Tensor:
    img = tf.image.random_brightness(img, 0.08)
    img = tf.image.random_contrast(img, 0.9, 1.1)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = random_rotate_180(img)
    img = random_sharpness(img)
    img = random_blur(img)
    img = add_gaussian_noise(img)
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img


def build_dataset(df: pd.DataFrame, preprocess_fn, training: bool) -> tf.data.Dataset:
    paths = df['image_path'].tolist()
    labels = df['label_idx'].tolist()
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=min(len(paths), 1000), seed=SEED, reshuffle_each_iteration=True)

    def _load(path, label):
        img = decode_image(path)
        if training:
            img = augment_image(img)
        img = preprocess_fn(img)
        return img, label

    ds = ds.map(_load, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

## Model Design and Training Strategy
Each backbone uses the same classifier head:
- GlobalAveragePooling2D
- Dropout(0.3)
- Dense(128, relu)
- Dropout(0.2)
- Dense(3, softmax)

Training is two-phase:
1. Feature extraction (frozen backbone, LR=1e-3)
2. Fine-tuning (unfreeze top layers, LR=1e-5)

In [ ]:
# Model factory and training helpers
BACKBONES = {
    'mobilenetv3': {
        'name': 'MobileNetV3Small',
        'builder': MobileNetV3Small,
        'preprocess': preprocess_mobilenetv3,
        'model_prefix': 'meatlens_mobilenetv3',
    },
    'efficientnetb0': {
        'name': 'EfficientNetB0',
        'builder': EfficientNetB0,
        'preprocess': preprocess_efficientnet,
        'model_prefix': 'meatlens_efficientnetb0',
    },
    'resnet50': {
        'name': 'ResNet50',
        'builder': ResNet50,
        'preprocess': preprocess_resnet50,
        'model_prefix': 'meatlens_resnet50',
    },
    'mobilenetv2': {
        'name': 'MobileNetV2',
        'builder': MobileNetV2,
        'preprocess': preprocess_mobilenetv2,
        'model_prefix': 'meatlens_mobilenetv2',
    },
}


def build_model(backbone_key: str) -> tuple[tf.keras.Model, tf.keras.Model]:
    cfg = BACKBONES[backbone_key]
    base = cfg['builder'](weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,))
    base.trainable = False
    inputs = layers.Input(shape=IMG_SIZE + (3,))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs, name=cfg['name'])
    return model, base


def compile_model(model: tf.keras.Model, lr: float) -> None:
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )


def set_fine_tune(base_model: tf.keras.Model) -> None:
    base_model.trainable = True
    total_layers = len(base_model.layers)
    fine_tune_at = max(1, int(total_layers * (1.0 - FINE_TUNE_FRACTION)))
    for i, layer in enumerate(base_model.layers):
        if i < fine_tune_at:
            layer.trainable = False
        else:
            if isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False


def train_two_phase(model: tf.keras.Model, base_model: tf.keras.Model, train_ds, val_ds, class_weights: dict, run_name: str):
    callbacks = [
        EarlyStopping(patience=8, restore_best_weights=True, monitor='val_loss'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4),
        ModelCheckpoint(filepath=f'checkpoints/{run_name}_best.keras', save_best_only=True, monitor='val_loss'),
    ]
    os.makedirs('checkpoints', exist_ok=True)

    compile_model(model, LR_FE)
    hist1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FE,
        callbacks=callbacks,
        class_weight=class_weights,
        verbose=1,
    )

    set_fine_tune(base_model)
    compile_model(model, LR_FT)
    hist2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FT,
        callbacks=callbacks,
        class_weight=class_weights,
        verbose=1,
    )
    return hist1, hist2


def plot_history(hist1, hist2, title: str):
    import matplotlib.pyplot as plt
    def _merge(metric):
        return list(hist1.history.get(metric, [])) + list(hist2.history.get(metric, []))
    metrics = ['loss', 'val_loss', 'accuracy', 'val_accuracy']
    data = {m: _merge(m) for m in metrics}
    epochs = range(1, len(data['loss']) + 1)
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    axs[0].plot(epochs, data['loss'], label='train')
    axs[0].plot(epochs, data['val_loss'], label='val')
    axs[0].set_title(f'{title} Loss')
    axs[0].set_xlabel('Epoch')
    axs[0].set_ylabel('Loss')
    axs[0].legend()
    axs[1].plot(epochs, data['accuracy'], label='train')
    axs[1].plot(epochs, data['val_accuracy'], label='val')
    axs[1].set_title(f'{title} Accuracy')
    axs[1].set_xlabel('Epoch')
    axs[1].set_ylabel('Accuracy')
    axs[1].legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Evaluation helpers
def evaluate_model(model: tf.keras.Model, test_ds, label_names: list[str], run_name: str, out_dir: str):
    if not HAVE_SKLEARN:
        raise RuntimeError('scikit-learn is required for detailed evaluation')
    os.makedirs(out_dir, exist_ok=True)
    y_true = []
    y_pred = []
    y_prob = []
    for batch_x, batch_y in test_ds:
        probs = model.predict(batch_x, verbose=0)
        preds = np.argmax(probs, axis=1)
        y_true.extend(batch_y.numpy().tolist())
        y_pred.extend(preds.tolist())
        y_prob.extend(probs.tolist())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    report = classification_report(y_true, y_pred, target_names=label_names, zero_division=0, output_dict=True)
    cm = confusion_matrix(y_true, y_pred)

    # Confusion matrix plot
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(f'Confusion Matrix: {run_name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_xticks(range(len(label_names)))
    ax.set_yticks(range(len(label_names)))
    ax.set_xticklabels(label_names)
    ax.set_yticklabels(label_names)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='black')
    fig.colorbar(im)
    plt.tight_layout()
    cm_path = os.path.join(out_dir, f'{run_name}_confusion_matrix.png')
    fig.savefig(cm_path)
    plt.show()
    plt.close(fig)

    # Save classification report
    report_path = os.path.join(out_dir, f'{run_name}_classification_report.csv')
    pd.DataFrame(report).transpose().to_csv(report_path)

    metrics = {
        'accuracy': acc,
        'precision_macro': precision,
        'recall_macro': recall,
        'f1_macro': f1,
        'cm_path': cm_path,
        'report_path': report_path,
    }
    return metrics, report, cm


def save_model_and_tflite(model: tf.keras.Model, prefix: str, exp: str):
    h5_path = f'{prefix}_{exp}.h5'
    tflite_path = f'{prefix}_{exp}.tflite'
    model.save(h5_path)
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)
    h5_size = os.path.getsize(h5_path) / (1024 * 1024)
    tflite_size = os.path.getsize(tflite_path) / (1024 * 1024)
    return h5_path, tflite_path, h5_size, tflite_size


def measure_inference_speed(model: tf.keras.Model, df_test: pd.DataFrame, preprocess_fn, n: int = 50) -> float:
    sample_df = df_test.sample(n=min(n, len(df_test)), random_state=SEED)
    paths = sample_df['image_path'].tolist()
    labels = sample_df['label_idx'].tolist()
    imgs = []
    for p in paths:
        img = decode_image(tf.constant(p))
        img = preprocess_fn(img)
        imgs.append(img)
    imgs = tf.stack(imgs, axis=0)
    # Warm-up
    _ = model.predict(imgs[:1], verbose=0)
    start = time.perf_counter()
    _ = model.predict(imgs, verbose=0)
    end = time.perf_counter()
    ms_per_image = (end - start) * 1000.0 / len(paths)
    return ms_per_image


def freshness_score(pred_label: str, confidence: float) -> tuple[int, str]:
    pred_label = pred_label.lower()
    if pred_label == 'fresh':
        score = 70 + int(round(confidence * 30))
    elif pred_label == 'not fresh':
        score = 40 + int(round(confidence * 29))
    else:
        score = int(round((1.0 - confidence) * 39))
    score = max(0, min(100, score))
    if score >= 70:
        rec = 'Good for Consumption'
    elif score >= 40:
        rec = 'Consume Immediately'
    else:
        rec = 'Not Suitable'
    return score, rec

In [ ]:
# Training and evaluation loop
results = []
histories = {}
model_store = {}
report_dir = Path('reports')
report_dir.mkdir(exist_ok=True)

for exp_name, splits in EXP_CSVS.items():
    print('\n=== Experiment:', exp_name, '===')
    train_df = add_label_index(load_split_csv(splits['train']), label_to_index)
    val_df = add_label_index(load_split_csv(splits['val']), label_to_index)
    test_df = add_label_index(load_split_csv(splits['test']), label_to_index)

    validate_paths(train_df)
    validate_paths(val_df)
    validate_paths(test_df)

    class_weights, counts = compute_class_weights(train_df['label_idx'].tolist(), num_classes)
    print('Class counts:', dict(zip(label_names, counts)))
    print('Class weights:', class_weights)

    for backbone_key, cfg in BACKBONES.items():
        run_name = f"{backbone_key}_{exp_name}"
        print('\nTraining:', run_name)
        preprocess_fn = cfg['preprocess']
        train_ds = build_dataset(train_df, preprocess_fn, training=True)
        val_ds = build_dataset(val_df, preprocess_fn, training=False)
        test_ds = build_dataset(test_df, preprocess_fn, training=False)

        model, base_model = build_model(backbone_key)
        hist1 = hist2 = None
        if RUN_TRAINING:
            hist1, hist2 = train_two_phase(model, base_model, train_ds, val_ds, class_weights, run_name)
            plot_history(hist1, hist2, f"{cfg['name']} {exp_name}")

        # Evaluate on test set only
        metrics, report, cm = evaluate_model(
            model, test_ds, label_names, run_name, str(report_dir)
        )

        # Save model and TFLite
        h5_path, tflite_path, h5_size, tflite_size = save_model_and_tflite(
            model, cfg['model_prefix'], exp_name
        )

        # Inference speed
        ms_per_image = measure_inference_speed(model, test_df, preprocess_fn, n=50)

        results.append({
            'experiment': exp_name,
            'backbone': cfg['name'],
            'backbone_key': backbone_key,
            'accuracy': metrics['accuracy'],
            'precision_macro': metrics['precision_macro'],
            'recall_macro': metrics['recall_macro'],
            'f1_macro': metrics['f1_macro'],
            'h5_path': h5_path,
            'tflite_path': tflite_path,
            'h5_size_mb': h5_size,
            'tflite_size_mb': tflite_size,
            'inference_ms': ms_per_image,
        })
        histories[(run_name, 'phase1')] = hist1
        histories[(run_name, 'phase2')] = hist2
        model_store[run_name] = model

results_df = pd.DataFrame(results)
results_df

In [ ]:
# Per-experiment metrics table
if not results_df.empty:
    display(results_df[['experiment', 'backbone', 'accuracy', 'precision_macro', 'recall_macro', 'f1_macro']])

# Average metrics across exp1 and exp2 for each backbone
avg_df = results_df.groupby(['backbone', 'backbone_key'])[['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']].mean().reset_index()
avg_df = avg_df.sort_values(by='f1_macro', ascending=False)
display(avg_df)

# Final comparison table (includes size and speed)
comparison_df = results_df.groupby(['backbone', 'backbone_key']).agg({
    'accuracy': 'mean',
    'precision_macro': 'mean',
    'recall_macro': 'mean',
    'f1_macro': 'mean',
    'h5_size_mb': 'mean',
    'tflite_size_mb': 'mean',
    'inference_ms': 'mean',
}).reset_index().sort_values(by='f1_macro', ascending=False)
display(comparison_df)

# Best model based on macro F1-score
best_row = comparison_df.iloc[0]
print('Best backbone by macro F1:', best_row['backbone'])
print(best_row)

In [ ]:
# Example single-image prediction with freshness scoring
def predict_single(model: tf.keras.Model, row: pd.Series, preprocess_fn):
    path = row['image_path']
    img = decode_image(tf.constant(path))
    img = preprocess_fn(img)
    img = tf.expand_dims(img, 0)
    probs = model.predict(img, verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    pred_label = label_names[pred_idx]
    confidence = float(probs[pred_idx])
    score, rec = freshness_score(pred_label, confidence)
    return {
        'image_path': path,
        'predicted_class': pred_label,
        'confidence': confidence,
        'freshness_score': score,
        'recommendation': rec,
    }

# Use best-performing backbone across experiments if available
if not results_df.empty:
    best_backbone_key = comparison_df.iloc[0]['backbone_key']
    # pick the best run from that backbone
    best_run = results_df[results_df['backbone_key'] == best_backbone_key].sort_values(by='f1_macro', ascending=False).iloc[0]
    run_key = f"{best_run['backbone_key']}_{best_run['experiment']}"
    model = model_store[run_key]
    preprocess_fn = BACKBONES[best_backbone_key]['preprocess']
    # Use first test image from exp1 for the demo
    demo_df = add_label_index(load_split_csv(EXP_CSVS['exp1']['test']), label_to_index)
    demo_row = demo_df.iloc[0]
    result = predict_single(model, demo_row, preprocess_fn)
    result

## Final Summary
- The best model is selected by **macro F1-score** (not accuracy).
- All metrics are computed on the test splits only.
- Confusion matrices and classification reports are stored in the `reports` folder.